## Notebook 3/3 — Cross-domain Transfer: DAIC-WOZ ↔ Reddit (Classical-only)

This notebook measures **out-of-domain generalization**:
- Train DAIC-WOZ (binary PHQ8) → Test Reddit (binary risk)
- Train Reddit (binary risk) → Test DAIC-WOZ (binary PHQ8)

**Prerequisite**: run `01_reddit_preprocess_embed.ipynb` first.

Outputs saved under `outputs/`:
- `daic_embeddings.npy`, `daic_labels_binary.npy`
- `transfer_daic_to_reddit.csv`, `transfer_reddit_to_daic.csv`
- `cm_transfer_*.png`

In [1]:
# Colab (optional)
!pip -q install -U pandas numpy scikit-learn sentence-transformers matplotlib seaborn

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pipeline_utils import (
    RANDOM_STATE,
    clean_text,
    generate_embeddings,
    build_models,
    evaluate_transfer,
)

sns.set_theme(style="whitegrid")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reddit artifacts
emb_path = OUTPUT_DIR / "reddit_embeddings.npy"
lab_path = OUTPUT_DIR / "reddit_labels.npy"
if not emb_path.exists() or not lab_path.exists():
    raise FileNotFoundError(
        "Missing Reddit artifacts. Run 01_reddit_preprocess_embed.ipynb first."
    )

reddit_X = np.load(emb_path)
reddit_y_3class = np.load(lab_path)
reddit_y_bin = (reddit_y_3class > 0).astype(np.int64)

print("Loaded Reddit embeddings:", reddit_X.shape)
print("Loaded Reddit labels (3-class):", reddit_y_3class.shape)
print("Derived Reddit binary labels distribution:", dict(zip(*np.unique(reddit_y_bin, return_counts=True))))


def save_fig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()


def plot_cm(cm: np.ndarray, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Not Depressed / Control", "Depressed / Risk"],
        yticklabels=["Not Depressed / Control", "Depressed / Risk"],
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    save_fig(out_path)


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Loaded Reddit embeddings: (4000, 384)
Loaded Reddit labels (3-class): (4000,)
Derived Reddit binary labels distribution: {np.int64(0): np.int64(2000), np.int64(1): np.int64(2000)}


### Load DAIC-WOZ transcripts + labels

Set the paths below to your DAIC transcript folder (contains `*_TRANSCRIPT.csv`) and label CSV.

If paths are missing, this notebook will stop with a helpful error.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA

OUTPUT_DIR = Path("outputs")

# Load train + dev splits
train_df = pd.read_csv(r"major 1\train_split_Depression_AVEC2017.csv")
dev_df = pd.read_csv(r"major 1\dev_split_Depression_AVEC2017.csv")
daic_labels_df = pd.concat([train_df, dev_df], ignore_index=True)

# Load pids and fused features
pids_df = pd.read_csv(r"major 1\pids.csv")
fused = np.load(r"major 1\fused_features.npy", allow_pickle=True).astype(np.float32)

# Match labels to participants
merged = pids_df.merge(
    daic_labels_df[["Participant_ID", "PHQ8_Binary"]], 
    on="Participant_ID", 
    how="inner"
)

print("Matched participants:", len(merged))
print("Label distribution:", merged["PHQ8_Binary"].value_counts().to_dict())
print("Fused features shape:", fused.shape)

Matched participants: 116
Label distribution: {0: 76, 1: 40}
Fused features shape: (154, 1152)


In [4]:
from sklearn.decomposition import PCA

n_components = min(116, fused.shape[1], len(merged))
pca = PCA(n_components=n_components, random_state=42)
daic_X = pca.fit_transform(fused[:len(merged)])
daic_y = merged["PHQ8_Binary"].to_numpy()

print("DAIC X shape after PCA:", daic_X.shape)
print("DAIC y shape:", daic_y.shape)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

np.save(OUTPUT_DIR / "daic_embeddings.npy", daic_X)
np.save(OUTPUT_DIR / "daic_labels_binary.npy", daic_y)
print("Saved DAIC embeddings and labels")

DAIC X shape after PCA: (116, 116)
DAIC y shape: (116,)
PCA explained variance: 1.000
Saved DAIC embeddings and labels


In [5]:
from pipeline_utils import build_models, evaluate_transfer
from sklearn.decomposition import PCA as PCA_r
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load Reddit artifacts
reddit_X = np.load(OUTPUT_DIR / "reddit_embeddings.npy")
reddit_y_bin = np.load(OUTPUT_DIR / "reddit_labels.npy").astype(np.int64)

# Reduce Reddit to same dims as DAIC
pca_reddit = PCA_r(n_components=116, random_state=42)
reddit_X_reduced = pca_reddit.fit_transform(reddit_X)
print("Reddit X reduced shape:", reddit_X_reduced.shape)
print("DAIC X shape:", daic_X.shape)

models = build_models()

def plot_cm(cm, title, out_path):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["MH Risk / Not Dep", "High Risk / Dep"],
                yticklabels=["MH Risk / Not Dep", "High Risk / Dep"], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

print("\n=== Transfer: Train DAIC-WOZ → Test Reddit ===")
results_d2r, cm_d2r = evaluate_transfer(
    daic_X, daic_y, reddit_X_reduced, reddit_y_bin, models
)
print(results_d2r[["model", "accuracy", "f1_weighted", "f1_macro"]])
results_d2r.to_csv(OUTPUT_DIR / "transfer_daic_to_reddit.csv", index=False)

for model_name, cm in cm_d2r.items():
    plot_cm(cm, f"{model_name} — Train:DAIC → Test:Reddit",
            OUTPUT_DIR / f"cm_transfer_daic_to_reddit_{model_name}.png")

print("Saved DAIC→Reddit results and confusion matrices")

Reddit X reduced shape: (4000, 116)
DAIC X shape: (116, 116)

=== Transfer: Train DAIC-WOZ → Test Reddit ===
  model  accuracy  f1_weighted  f1_macro
0    LR    0.5045     0.504498  0.504498
2   MLP    0.5045     0.504498  0.504498
1   SVM    0.5000     0.333333  0.333333
Saved DAIC→Reddit results and confusion matrices


In [6]:
print("=== Transfer: Train Reddit → Test DAIC-WOZ ===")
results_r2d, cm_r2d = evaluate_transfer(
    reddit_X_reduced, reddit_y_bin, daic_X, daic_y, models
)
print(results_r2d[["model", "accuracy", "f1_weighted", "f1_macro"]])
results_r2d.to_csv(OUTPUT_DIR / "transfer_reddit_to_daic.csv", index=False)

for model_name, cm in cm_r2d.items():
    plot_cm(cm, f"{model_name} — Train:Reddit → Test:DAIC",
            OUTPUT_DIR / f"cm_transfer_reddit_to_daic_{model_name}.png")

print("Saved Reddit→DAIC results and confusion matrices")

=== Transfer: Train Reddit → Test DAIC-WOZ ===
  model  accuracy  f1_weighted  f1_macro
2   MLP  0.577586     0.587357  0.560300
0    LR  0.560345     0.567420  0.531034
1   SVM  0.543103     0.551827  0.516933
Saved Reddit→DAIC results and confusion matrices


In [7]:
results_d2r["Direction"] = "DAIC → Reddit"
results_r2d["Direction"] = "Reddit → DAIC"
transfer_summary = pd.concat([results_d2r, results_r2d], ignore_index=True)

print("\n=== Cross-Domain Transfer Summary ===")
print(transfer_summary[["Direction", "model", "accuracy", "f1_weighted", "f1_macro"]].to_string(index=False))
transfer_summary.to_csv(OUTPUT_DIR / "transfer_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=transfer_summary, x="model", y="f1_weighted", hue="Direction", ax=ax)
ax.set_title("Cross-Domain Transfer: Weighted F1 by Model and Direction")
ax.set_xlabel("Model")
ax.set_ylabel("Weighted F1")
ax.legend(title="Transfer Direction")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "transfer_f1_comparison.png", dpi=200, bbox_inches="tight")
plt.close()

print("All done! Saved transfer_summary.csv and transfer_f1_comparison.png")


=== Cross-Domain Transfer Summary ===
    Direction model  accuracy  f1_weighted  f1_macro
DAIC → Reddit    LR  0.504500     0.504498  0.504498
DAIC → Reddit   MLP  0.504500     0.504498  0.504498
DAIC → Reddit   SVM  0.500000     0.333333  0.333333
Reddit → DAIC   MLP  0.577586     0.587357  0.560300
Reddit → DAIC    LR  0.560345     0.567420  0.531034
Reddit → DAIC   SVM  0.543103     0.551827  0.516933
All done! Saved transfer_summary.csv and transfer_f1_comparison.png
